```java
import org.spongepowered.asm.mixin.Mixin;
import org.spongepowered.asm.mixin.Shadow;

@Mixin(EntityPlayer.class)
public abstract class MixinEntityPlayer extends Entity implements LivingThing, Leveler {
    // Пример Shadow для поля
    @Shadow
    private int level;

    // Пример Shadow для метода
    @Shadow
    private void update();

    // Пример реализации метода интерфейса подключенного к mixins классу
    @Override
    public void setLevel(int newLevel) {
        // Обращение к shadow-полю; при применении mixin это будет ссылкой на реальное поле целевого класса
        this.level = newLevel;
    }
}
```

![Пример mixins](../../resources/mixin_tut_8.png)

Целевой класс (`EntityPlayer.class`) — это класс, в который встраивается **mixins** класс (указан в `@Mixin`)

`@Mixin(Target.class)` — указывает целевой класс для **mixins** класса

`@Shadow` — делает поле или метод видимым в **mixins** классе, но не создаёт его — сигнатуры должны совпадать с целевыми, **Shadow**-методы не должны иметь тела (объявляются без реализации)

`@Override` — реализация, внедрённого с помощью **mixins**, интерфейса

```java
@Mixin(Bar.class)
@Implements(@Interface(iface = Indentifyable.class, prefix = "ident$"))
public abstract class MixinBar extends Foo {

    private final UUID id = UUID.randomUUID();

    public UUID ident$getID() {
        return this.id;
    }
}
```
# ![Пример softimplementation](../../resources/mixin_conflict_3.png)

`@Implements` — позволяет реализовать интерфейс в **mixins**-классе, добавляя префикс к методам интерфейса.

`softimplementation` — это способ переопределить метод в целевом классе, если существует метод с такой же сигнатурой в родительском классе или интерфейсе, хотя есть и совершенно другой подход

```java
public abstract class MixinFoo implements Identifyable {
    @Shadow
    public abstract int shadow$getID();

    @Shadow(prefix = "conflict$")
    public abstract int conflict$getID();

    public UUID getID() {
        // return the unique ID
    }
}
```
![Пример softshadow](../../resources/mixin_conflict_4.png)

`@Shadow(prefix = "conflict$")` — позволяет избежать конфликтов имен, добавляя префикс к сгенерированным методам.

```java
@Mixin(EntityPlayer.class)
public abstract class MixinEntityPlayer
    extends Entity
    implements LivingThing, Leveller {

    @Shadow
    private int level;

    /**
     * Adding the annotation to our overwrite connects it with
     * its will-be-obfuscated target method.
     */
    @Overwrite
    public int getLevel() {
        return Math.max(this.level, 0);
    }

    @Override
    public void setLevel(int newLevel) {
        this.level = Math.max(newLevel, 0);
    }
}
```

![Пример overwrite](../../resources/mixin_tut_22.png)

`@Overwrite` — аннотация, которая позволяет полностью переопределить метод целевого класса; стоит всячески избегать использования `@Overwrite`, если только это не абсолютно необходимо, безопасная альтернатива — использование `@Inject` для добавления дополнительной логики.

```java
@Overwrite(constraints = "BUILD(1234)")
public void someHackyOverwrite(int x, int y) {
    // do hacky things
}
```

`constraints` — это параметр, который позволяет указать условия, при которых будет применяться переопределение. Например, `BUILD(1234)` означает, что переопределение будет применяться только в сборке с номером `1234`, также возможны следующие условия

![Таблица с типами constraints](../../resources/flard_17.jpg)

```java
@Mixin(Foo.class)
@Implements(@Interface(iface = Indentifyable.class, prefix = "id$"))
public abstract class MixinFoo {
    @Shadow
    public abstract int getID();

    /**
     * This method will become our intrinsic proxy method, it
     * calls the original (shadowed) version of the accessor.
     * It uses the displace parameter to avoid re-entrance when
     * the method would otherwise be overwritten.
     */
    @Intrinsic(displace = true)
    public int id$getID() {
        // Call original accessor
        return this.getID();
    }
}
```

![Пример intrinsics](../../resources/mixin_tut_29.png)

`@Intrinsic(displace = true)` — аннотация, которая позволяет создать прокси-метод для оригинального метода, которая гарантирует единственное вхождение созданного метода в целевом классе

```java
@Inject(method = "update", at = @At("HEAD"))
protected void onUpdate(CallbackInfo ci) {
    Observer.instance.foo(this);
}
```

![Пример injection](../../resources/flard_16.png)

`@Inject` — вставляет код в указанную точку определённого метода

`@At("HEAD")` — указывает, что код будет вызван как первая процедура метода

```java
/**
 * Handler method, onSetPos. Note the two int variables x and y
 * which appear before the callbackinfo
 */
@Inject(method = "setPos", at = @At("HEAD"))
protected void onSetPos(int x, int y, CallbackInfo ci) {
    System.out.printf("Position is being set to (%d, %d)\n", x, y);
}
```

![Пример injection](../../resources/flard_06.png)

Позволяет получать переменные в качестве аргументов

```java
/**
 * Cancellable injection, note that we set the "cancellable"
 * flag to "true" in the injector annotation
 */
@Inject(method = "setPos", at = @At("HEAD"), cancellable = true)
protected void onSetPos(int x, int y, CallbackInfo ci) {
    // Check whether setting position to origin and do some custom logic
    if (x == 0 && y == 0) {
        // Some custom logic
        this.position = Point.ORIGIN;
        this.handleOriginPosition();
        
        // Call update() just like the original method would have
        this.update();
        
        // Mark the callback as cancelled
        ci.cancel();
    }
    
    // Execution proceeds as normal at this point, no custom handling
}
```

![Пример injection](../../resources/flard_07.png)

Позволяет прерывать исполнение основного метода

```java
@Inject(method = "setPos", at = @At(value = "RETURN", ordinal = 0))
protected void onResetPos(int x, int y, CallbackInfo ci) {
    // handler logic
}
```

![Пример injection](../../resources/flard_09.png)

Позволяет указывать номер вхождения опкода в случае наличия множества вхождений указанного опкода в целевом методе

`@At("RETURN")` — указывает, что код будет вставлен перед инструкцией `return`

Как аргумент аннотации `@At` можно использовать некоторое количество ключевых слов позволяющих найти ключевые опкоды из байт кода, которые присутствуют в целевом методе, список ключевых

`@At("TAIL")` — указывает, что код будет вставлен в конец метода перед последней инструкцией

`@At(value = "INVOKE", target = "<method>")` — находит вызов обозначенного метода и вставляет код перед ним

`@At("FIELD")` — находит поле, которое может прочитать или переписать, и встраивает код перед его первым вхождением (ищет вхождения инструкций `GETFIELD` и `PUTFIELD`)

`@At("NEW")` — находит инструкцию создания нового объекта (ключевое слово `new`) и встраивает код перед ней

`@At("EXCEPTION")` — указывает, что код будет вставлен в обработчик исключений целевого метода (внутри блока catch) и выполнится при возникновении исключения во время выполнения метода

`@At("JUMP")` — указывает, что код будет вставлен перед инструкцией перехода (охватывает все IF_* (условные), GOTO (безусловный), SWITCH (TABLE/LOOKUP) инструкции) и выполнится перед выполнением перехода

`@At("INVOKE_STRING")` — находит вызов метода с типом `String` в качестве единственного аргумента и возвращаемым значением `void`, и вставляет код перед ним, полезно для встраивания кода перед вызовом `Profiler.startSection(nameOfSection)`

`@At(value = "INVOKE_ASSIGN", target = "<method>")` — находит вызов обозначенного метода, который возвращает значение (не `void`), и вставляет код ПОСЛЕ него (единственная точка вставки, которая вставляет код после вызова инструкции)

```java
@Inject(method = "getPos", at = @At("HEAD"), cancellable = true)
protected void onGetPos(CallbackInfoReturnable<Point> cir) {
    if (this.position == null) {
        // setReturnValue implicitly cancel()s the callback
        cir.setReturnValue(Point.ORIGIN);
    }
    
    // Note that if the handler returns normally then the method
    // continues as normal, just like a normal cancellable
    //  injection does if cancel() is not called.
}
```

![Пример injection](../../resources/flard_12.png)

`CallbackInfoReturnable` —  generic класс, вариант `CallbackInfo`, который позволяет устанавливать возвращаемое значение метода 

```java
@Inject(method = "getPos", at = @At("RETURN"), cancellable = true)
protected void onGetPos(CallbackInfoReturnable<Point> cir) {
    // Check the captured return value
    if (cir.getReturnValue() == null) {
        // if it's null, set our fallback value as the return
        cir.setReturnValue(Point.ORIGIN);
    }
}
```

![Пример injection](../../resources/flard_14.png)

Если код метода mixins встраивается перед инструкцией `return`, то обработчик захватывает значение и помещает его в `CallbackInfoReturnable`, что позволяет изменять возвращаемое значение метода

```java
@Inject(method = "<init>*", at = @At("RETURN"))
private void onConstructed(CallbackInfo ci) {
    // do initialisation stuff
}
```

Пример выше встраивает код в каждый конструктор класса (в конструкторы класса можно встраиваться только с аннотацией `@At("RETURN")`), использование `method = "*"` охватывает все методы класса, единственное стоит учитывать, что если для метода с `void` возвращаемым значением использовать в аргументе `CallbackInfoReturnable`, то это приведет к ошибке компиляции, так как `CallbackInfoReturnable` ожидает возвращаемое значение

```java
@Inject(method = "foo", at = @At(value = "INVOKE", target = "someMethod"), require = 2)
private void onInvokeSomeMethodInFoo(CallbackInfo ci) {
    ...
```

Параметр `at` является массивом, поэтому возможна множественная инъекция, параметр `require` указывает минимальное количество вхождений указанного опкода, необходимых для выполнения инъекции, иначе будет выброшено исключение `InjectionError`

### Useful repositories with comments
https://github.com/CaffeineMC/sodium
https://github.com/CaffeineMC/lithium
### Useful documentations
https://gist.github.com/Mumfrey/000438f56f23b91c3276d4deba4a412e
https://gist.github.com/TelepathicGrunt/3784f8a8b317bac11039474012de5fb4
https://github.com/dblsaiko/mixin-cheatsheet?tab=readme-ov-file

### Annotations
`@Accessor` — генерирует геттер/сеттер для приватного поля (рекомендуется вместо прямого `@shadow`, когда нужен доступ к значению)

`@Invoker` — генерирует метод для вызова приватного метода целевого класса